# Auth Log Triage — SanyaKalra

**Purpose:** Parse and analyse a real-world-style OpenSSH `auth.log` file to surface authentication anomalies such as brute-force attempts, invalid-user probes, and legitimate login activity.

**Data files (pre-existing, not created by this notebook):**
- `data/sample_auth.log` — synthetic OpenSSH syslog entries (Oct 18 2026)
- `data/mock_reputation.json` — synthetic IP reputation scores (for later sections)

---

## Section 1 — Load & Parse `sample_auth.log`

This section:
1. Reads the raw log file line-by-line.
2. Uses **three targeted regular expressions** to identify:
   - `failed_password` — a valid-but-wrong password attempt (`Failed password for <user>`)
   - `accepted_password` — a successful login (`Accepted password for <user>`)
   - `invalid_user` — a probe for a username that does not exist on the host (`Invalid user <user>`)
3. Extracts **timestamp** (year hard-set to 2026), **event type**, **username**, and **source IP** into a `pandas` DataFrame.
4. Lines that match **none** of the three patterns are counted and skipped (e.g. `pam_unix` session-open messages).

> **Note on overlap:** OpenSSH sometimes logs both `Invalid user` *and* `Failed password for invalid user` for the same probe connection. Both events are captured separately because they represent distinct log lines and different analytical signals.

In [29]:
# Standard-library and third-party imports for this section
import re
from datetime import datetime
import pandas as pd

### 1.1 — Regex patterns

Each pattern targets one event type. All three share the same syslog header prefix:
```
Mon DD HH:MM:SS hostname sshd[pid]:
```
Named capture groups (`?P<name>`) are used so extracted fields read by name rather than positional index.

In [30]:
# Shared syslog header: month, day, time — e.g. "Oct 18 02:14:00"
# This prefix is reused in every pattern below.
SYSLOG_PREFIX = r"(?P<month>\w{3})\s+(?P<day>\d{1,2})\s+(?P<time>\d{2}:\d{2}:\d{2})"

# Pattern 1 — Failed password
# Matches lines like:
#   "Failed password for admin from 203.0.113.45 port 61631 ssh2"
#   "Failed password for invalid user oracle from 192.0.2.10 port 37395 ssh2"
# The optional "invalid user" segment is captured so we can normalise the username.
PAT_FAILED = re.compile(
    SYSLOG_PREFIX
    + r"\s+\S+\s+sshd\[\d+\]: Failed password for (?:invalid user )?(?P<user>\S+)"
      r" from (?P<ip>[\d.]+)"
)

# Pattern 2 — Accepted password
# Matches lines like:
#   "Accepted password for rahul from 10.0.0.22 port 34578 ssh2"
PAT_ACCEPTED = re.compile(
    SYSLOG_PREFIX
    + r"\s+\S+\s+sshd\[\d+\]: Accepted password for (?P<user>\S+)"
      r" from (?P<ip>[\d.]+)"
)

# Pattern 3 — Invalid user (notification line, before password attempt)
# Matches lines like:
#   "Invalid user test from 192.0.2.10 port 31185"
PAT_INVALID = re.compile(
    SYSLOG_PREFIX
    + r"\s+\S+\s+sshd\[\d+\]: Invalid user (?P<user>\S+)"
      r" from (?P<ip>[\d.]+)"
)

# Convenience list used by the parser: (event_label, compiled_pattern)
PATTERNS = [
    ("failed_password",  PAT_FAILED),
    ("accepted_password", PAT_ACCEPTED),
    ("invalid_user",     PAT_INVALID),
]

### 1.2 — Line parser

`parse_line()` tries each pattern in order and returns a structured dict on the first match, or `None` if the line is unrecognised.

In [31]:
def parse_line(line: str, year: int = 2026) -> dict | None:
    """
    Try every (event_type, pattern) pair against a single log line.

    Parameters
    ----------
    line : str
        One raw line from the auth log (trailing newline is fine).
    year : int
        Year to inject when building the datetime object; syslog headers
        omit the year, so we assume all entries belong to `year`.

    Returns
    -------
    dict with keys {timestamp, event_type, username, source_ip}
    or None if no pattern matched.
    """
    for event_type, pattern in PATTERNS:
        m = pattern.search(line)        # search() handles leading whitespace / BOM
        if m:
            # Build a full datetime from the syslog month/day/time + injected year
            raw_dt = f"{year} {m['month']} {m['day']:>2} {m['time']}"
            timestamp = datetime.strptime(raw_dt, "%Y %b %d %H:%M:%S")
            return {
                "timestamp":  timestamp,
                "event_type": event_type,
                "username":   m["user"],
                "source_ip":  m["ip"],
            }
    return None     # line matched no known event type

### 1.3 — Load the log file and build the DataFrame

`load_auth_log()` reads the file, calls `parse_line()` on every line, accumulates parsed records, and reports the count of skipped lines to stdout.

In [32]:
def load_auth_log(path: str, year: int = 2026) -> pd.DataFrame:
    """
    Read an OpenSSH auth log and return a tidy DataFrame.

    Parameters
    ----------
    path : str
        Filesystem path to the .log file.
    year : int
        Year to assume for all log timestamps (default 2026).

    Returns
    -------
    pd.DataFrame
        Columns: timestamp (datetime64), event_type (str),
                 username (str), source_ip (str).
        Rows are in original log order.
    """
    records = []    # accumulate successfully parsed rows
    skipped = 0     # count of lines that matched no pattern

    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n")    # strip newline before matching
            if not line.strip():        # skip blank lines silently
                continue

            result = parse_line(line, year=year)
            if result:
                records.append(result)
            else:
                skipped += 1            # count unmatched lines for reporting

    print(f"Parsed : {len(records):>4} matching lines")
    print(f"Skipped: {skipped:>4} non-matching lines")

    # Convert list-of-dicts to DataFrame and enforce correct dtypes
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df

In [33]:
# Run the loader — reads from the pre-existing data file
LOG_PATH = "data/sample_auth.log"

df = load_auth_log(LOG_PATH, year=2026)

Parsed :  128 matching lines
Skipped:   50 non-matching lines


### 1.4 — Inspect the result

Quick sanity checks: shape, dtypes, event-type distribution, and the first few rows.

In [34]:
# Overall shape and column types
print(f"DataFrame shape: {df.shape}")
print()
print(df.dtypes)

DataFrame shape: (128, 4)

timestamp     datetime64[ns]
event_type            object
username              object
source_ip             object
dtype: object


In [35]:
# Count of each event type — confirms all three categories were captured
print("Event type counts:")
print(df["event_type"].value_counts())

Event type counts:
event_type
failed_password      69
accepted_password    50
invalid_user          9
Name: count, dtype: int64


In [36]:
# Preview the first 10 rows with all four columns
df.head(10)

,timestamp,event_type,username,source_ip
0,2026-10-18 02:14:00,failed_password,admin,203.0.113.45
1,2026-10-18 02:14:04,failed_password,admin,203.0.113.45
2,2026-10-18 02:14:09,failed_password,admin,203.0.113.45
3,2026-10-18 02:14:14,failed_password,admin,203.0.113.45
4,2026-10-18 02:14:19,failed_password,admin,203.0.113.45
5,2026-10-18 02:14:24,failed_password,admin,203.0.113.45
6,2026-10-18 02:14:29,failed_password,admin,203.0.113.45
7,2026-10-18 02:14:34,failed_password,admin,203.0.113.45
8,2026-10-18 02:14:38,failed_password,admin,203.0.113.45
9,2026-10-18 02:14:43,failed_password,admin,203.0.113.45


In [37]:
# Confirm timestamp range — should span 2026-10-18 only
print("Earliest:", df["timestamp"].min())
print("Latest:  ", df["timestamp"].max())

Earliest: 2026-10-18 02:14:00
Latest:   2026-10-18 19:44:19


In [38]:
# Unique source IPs seen across all event types
print("Source IPs:")
print(df["source_ip"].value_counts())

Source IPs:
source_ip
203.0.113.45     40
192.0.2.10       18
10.0.0.31        17
10.0.0.15        16
198.51.100.77    15
10.0.0.22        14
10.0.0.5          8
Name: count, dtype: int64


---
## Section 2 — Rule-Based Anomaly Detection

Three detection rules are applied directly to `df` built in Section 1. Each rule is a self-contained function that accepts the DataFrame and the tunable thresholds defined at the top of this section.

| Rule | What it flags |
|---|---|
| `rule_burst` | >= N `failed_password` events from one IP inside any 5-minute sliding window |
| `rule_success_after_failures` | An `accepted_password` within T minutes of >= F prior failures from the same IP |
| `rule_many_usernames` | >= U distinct usernames probed by one IP across `failed_password` + `invalid_user` events |

**How sliding windows work here:** for `rule_burst`, each failure event is treated as a window *start*; we count how many subsequent failures from the same IP fall within the next 5 minutes. If that count reaches the threshold, the IP is flagged. This is O(n^2) in the worst case but correct and readable for log-sized data.

> All thresholds are variables — change them in the **2.1 Thresholds** cell and re-run.

### 2.1 — Thresholds (edit here to tune all rules)

In [39]:
# ── Rule 1: burst detection ──────────────────────────────────────────────────
BURST_MIN_EVENTS  = 10   # minimum failures inside the window to trigger
BURST_WINDOW_MINS =  5   # sliding window width in minutes

# ── Rule 2: success after failures ──────────────────────────────────────────
SAF_MIN_FAILURES  =  5   # minimum prior failures required
SAF_LOOKBACK_MINS = 10   # how far back (minutes) to look for those failures

# ── Rule 3: username enumeration ────────────────────────────────────────────
ENUM_MIN_USERNAMES = 5   # minimum distinct usernames to trigger

### 2.2 — Confirm column names

Before writing any rule logic, assert that `df` has the four columns produced by Section 1. This makes the notebook self-documenting and catches any re-ordering issues early.

In [40]:
# Guard: verify the expected columns are present before running detection
EXPECTED_COLS = {"timestamp", "event_type", "username", "source_ip"}
assert EXPECTED_COLS.issubset(df.columns), (
    f"Missing columns: {EXPECTED_COLS - set(df.columns)}"
)
print("Column check passed. Columns:", list(df.columns))
print("Total rows:", len(df))

Column check passed. Columns: ['timestamp', 'event_type', 'username', 'source_ip']
Total rows: 128


### 2.3 — Rule 1: `rule_burst` — rapid failure burst from one IP

**Logic:** isolate `failed_password` rows, sort by time, then for every failure event check how many other failures from the *same IP* occur within the next `BURST_WINDOW_MINS` minutes. If any window contains >= `BURST_MIN_EVENTS` events, the IP is flagged.

**Expected trigger:** `203.0.113.45` fires 40 failures in ~2 minutes; `198.51.100.77` fires 14 in ~2 minutes.

In [41]:
from datetime import timedelta

def rule_burst(df, min_events=BURST_MIN_EVENTS, window_mins=BURST_WINDOW_MINS):
    """
    Detect IPs that produce >= min_events failed_password events
    within any sliding window of window_mins minutes.

    Returns a list of dicts, one per triggering IP, with:
      ip, max_count (peak window size), first_seen, last_seen.
    """
    # Keep only failure events and sort chronologically
    failures = (
        df[df["event_type"] == "failed_password"]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    window = timedelta(minutes=window_mins)   # reusable timedelta for comparisons
    hits = {}   # ip -> peak count seen in any single window

    # Group by IP so we only compare events from the same source
    for ip, grp in failures.groupby("source_ip"):
        times = grp["timestamp"].tolist()   # sorted list of failure timestamps for this IP
        peak = 0
        for i, t_start in enumerate(times):
            # Count how many failures fall within [t_start, t_start + window]
            count = sum(1 for t in times[i:] if t - t_start <= window)
            if count > peak:
                peak = count
        if peak >= min_events:
            hits[ip] = {
                "ip":         ip,
                "max_count":  peak,
                "first_seen": grp["timestamp"].min(),
                "last_seen":  grp["timestamp"].max(),
            }

    return list(hits.values())

In [42]:
# Run rule_burst and print results
burst_results = rule_burst(df)

print(f"rule_burst  (>= {BURST_MIN_EVENTS} failures in {BURST_WINDOW_MINS} min)")
print("-" * 60)
if burst_results:
    for r in burst_results:
        print(
            f"  IP: {r['ip']:<18}  "
            f"peak={r['max_count']:>3} events  "
            f"from {r['first_seen'].strftime('%H:%M:%S')} "
            f"to {r['last_seen'].strftime('%H:%M:%S')}"
        )
else:
    print("  No IPs triggered this rule.")

rule_burst  (>= 10 failures in 5 min)
------------------------------------------------------------
  IP: 198.51.100.77       peak= 14 events  from 14:30:00 to 14:31:49
  IP: 203.0.113.45        peak= 40 events  from 02:14:00 to 02:16:57


### 2.4 — Rule 2: `rule_success_after_failures` — login succeeds after repeated failures

**Logic:** for every `accepted_password` event, look back `SAF_LOOKBACK_MINS` minutes and count `failed_password` events from the *same IP*. If that count >= `SAF_MIN_FAILURES`, the login is suspicious (possible brute-force success).

**Expected trigger:** `198.51.100.77` has 14 failures followed by an accepted login for `root`.

In [43]:
def rule_success_after_failures(
    df,
    min_failures=SAF_MIN_FAILURES,
    lookback_mins=SAF_LOOKBACK_MINS,
):
    """
    Find accepted_password events that are preceded by >= min_failures
    failed_password events from the same source IP within lookback_mins minutes.

    Returns a list of dicts, one per suspicious login:
      ip, username, login_time, prior_failures (count), earliest_failure.
    """
    lookback = timedelta(minutes=lookback_mins)   # window reaching back from the login

    # Separate the two event types up front for clarity
    successes = df[df["event_type"] == "accepted_password"].copy()
    failures  = df[df["event_type"] == "failed_password"].copy()

    alerts = []
    for _, row in successes.iterrows():
        ip         = row["source_ip"]
        login_time = row["timestamp"]

        # Failures from this IP within the lookback window ending at login_time
        window_failures = failures[
            (failures["source_ip"] == ip)
            & (failures["timestamp"] >= login_time - lookback)
            & (failures["timestamp"] <  login_time)   # must precede the login
        ]

        if len(window_failures) >= min_failures:
            alerts.append({
                "ip":               ip,
                "username":         row["username"],
                "login_time":       login_time,
                "prior_failures":   len(window_failures),
                "earliest_failure": window_failures["timestamp"].min(),
            })

    return alerts

In [44]:
# Run rule_success_after_failures and print results
saf_results = rule_success_after_failures(df)

print(
    f"rule_success_after_failures  "
    f"(>= {SAF_MIN_FAILURES} failures in {SAF_LOOKBACK_MINS} min before login)"
)
print("-" * 60)
if saf_results:
    for r in saf_results:
        print(
            f"  IP: {r['ip']:<18}  user={r['username']:<12}  "
            f"login={r['login_time'].strftime('%H:%M:%S')}  "
            f"prior_failures={r['prior_failures']}  "
            f"(earliest failure {r['earliest_failure'].strftime('%H:%M:%S')})"
        )
else:
    print("  No IPs triggered this rule.")

rule_success_after_failures  (>= 5 failures in 10 min before login)
------------------------------------------------------------
  IP: 198.51.100.77       user=root          login=14:32:20  prior_failures=14  (earliest failure 14:30:00)


### 2.5 — Rule 3: `rule_many_usernames` — username enumeration

**Logic:** pool `failed_password` and `invalid_user` events (both indicate a rejected authentication attempt), group by `source_ip`, and count distinct usernames. An IP that tries >= `ENUM_MIN_USERNAMES` different usernames is likely running a credential-stuffing or user-enumeration scan.

**Expected trigger:** `192.0.2.10` probes 9 distinct non-existent usernames (`test`, `oracle`, `ubuntu`, `postgres`, `git`, `guest`, `user`, `pi`, `ftpuser`).

In [45]:
def rule_many_usernames(df, min_usernames=ENUM_MIN_USERNAMES):
    """
    Detect IPs that probe >= min_usernames distinct usernames
    across failed_password and invalid_user events.

    Returns a list of dicts:
      ip, unique_usernames (count), usernames (sorted list),
      first_seen, last_seen.
    """
    # Combine the two event types that represent rejected/probed authentication
    probe_events = df[
        df["event_type"].isin(["failed_password", "invalid_user"])
    ].copy()

    alerts = []
    for ip, grp in probe_events.groupby("source_ip"):
        unique_users = grp["username"].unique()   # ndarray of distinct usernames
        if len(unique_users) >= min_usernames:
            alerts.append({
                "ip":               ip,
                "unique_usernames": len(unique_users),
                "usernames":        sorted(unique_users.tolist()),
                "first_seen":       grp["timestamp"].min(),
                "last_seen":        grp["timestamp"].max(),
            })

    # Sort output by descending unique-username count so worst offenders appear first
    alerts.sort(key=lambda x: x["unique_usernames"], reverse=True)
    return alerts

In [46]:
# Run rule_many_usernames and print results
enum_results = rule_many_usernames(df)

print(f"rule_many_usernames  (>= {ENUM_MIN_USERNAMES} distinct usernames per IP)")
print("-" * 60)
if enum_results:
    for r in enum_results:
        print(
            f"  IP: {r['ip']:<18}  "
            f"unique_users={r['unique_usernames']}  "
            f"from {r['first_seen'].strftime('%H:%M:%S')} "
            f"to {r['last_seen'].strftime('%H:%M:%S')}"
        )
        print(f"    usernames: {', '.join(r['usernames'])}")
else:
    print("  No IPs triggered this rule.")

rule_many_usernames  (>= 5 distinct usernames per IP)
------------------------------------------------------------
  IP: 192.0.2.10          unique_users=9  from 05:20:00 to 05:24:21
    usernames: ftpuser, git, guest, oracle, pi, postgres, test, ubuntu, user


### 2.6 — Combined summary

Collect every IP that triggered at least one rule into a single overview table.

In [47]:
# Build a summary DataFrame: one row per IP, one column per rule
all_flagged_ips = (
    {r["ip"] for r in burst_results}
    | {r["ip"] for r in saf_results}
    | {r["ip"] for r in enum_results}
)

# Index the results by IP for fast lookup
burst_ips = {r["ip"]: r["max_count"]        for r in burst_results}
saf_ips   = {r["ip"]: r["prior_failures"]   for r in saf_results}
enum_ips  = {r["ip"]: r["unique_usernames"]  for r in enum_results}

summary_rows = []
for ip in sorted(all_flagged_ips):
    summary_rows.append({
        "source_ip":            ip,
        "burst_peak":           burst_ips.get(ip, "-"),    # peak window count or '-'
        "saf_prior_failures":   saf_ips.get(ip,   "-"),   # failures before login or '-'
        "enum_unique_users":    enum_ips.get(ip,   "-"),   # unique usernames or '-'
    })

summary_df = pd.DataFrame(summary_rows)
print("IPs flagged by at least one rule:")
summary_df

IPs flagged by at least one rule:


,source_ip,burst_peak,saf_prior_failures,enum_unique_users
0,192.0.2.10,-,-,9
1,198.51.100.77,14,14,-
2,203.0.113.45,40,-,-


---
## Section 3 — Enrichment, Severity & Verdict

This section takes every IP flagged by any rule in Section 2, enriches it with reputation data from `data/mock_reputation.json`, and produces a structured alert table with:

| Field | Description |
|---|---|
| **ip** | Source IP address |
| **rules_triggered** | Which detection rules fired |
| **evidence** | Counts, time window, usernames tried, success time if applicable |
| **abuse_confidence** | Score from reputation feed (0–100) or "no data" |
| **reports** | Number of external abuse reports or "no data" |
| **severity** | High / Medium / Low — derived from a point-based scoring model |
| **verdict** | Analyst recommendation and MITRE ATT&CK technique(s) |

### Severity scoring

| Contributor | Points |
|---|---|
| `rule_burst` triggered | +2 |
| `rule_success_after_failures` triggered | +4 |
| `rule_many_usernames` triggered | +2 |
| `abuse_confidence` ≥ 75 | +2 |
| `abuse_confidence` 30–74 | +1 |

**Thresholds:** High ≥ 6 points · Medium 3–5 · Low < 3

### 3.1 — Load reputation data

`data/mock_reputation.json` is loaded as a plain `dict`. Keys starting with `_` (metadata/comments) are skipped so only IP entries are retained.

In [48]:
import json

REPUTATION_PATH = "data/mock_reputation.json"

with open(REPUTATION_PATH, "r", encoding="utf-8") as _fh:
    _raw_rep = json.load(_fh)

# Ignore keys that start with '_' (metadata/notes in the file)
reputation = {k: v for k, v in _raw_rep.items() if not k.startswith("_")}

print(f"Loaded reputation data for {len(reputation)} IP(s):")
for _ip, _data in reputation.items():
    print(f"  {_ip:<18}  abuse_confidence={_data['abuse_confidence']}  reports={_data['reports']}")

Loaded reputation data for 4 IP(s):
  203.0.113.45        abuse_confidence=92  reports=310
  198.51.100.77       abuse_confidence=78  reports=140
  192.0.2.10          abuse_confidence=35  reports=22
  10.0.0.15           abuse_confidence=0  reports=0


### 3.2 — Build alert rows for flagged IPs

For every IP in `all_flagged_ips` (from Section 2.6) we collect:
- Which rules it triggered
- Evidence details from those rule results
- Reputation values (or `"no data"` when the IP is not in the reputation feed)

In [49]:
# ── Helper: pull the full rule result dict for an IP ────────────────────────
# Each rule function returns a list; index by IP for fast look-up.
_burst_by_ip = {r["ip"]: r for r in burst_results}
_saf_by_ip   = {r["ip"]: r for r in saf_results}
_enum_by_ip  = {r["ip"]: r for r in enum_results}

def _build_evidence(ip):
    """Compose a human-readable evidence string from whichever rules fired."""
    parts = []

    if ip in _burst_by_ip:
        b = _burst_by_ip[ip]
        parts.append(
            f"burst: {b['max_count']} failures in 5-min window "
            f"({b['first_seen'].strftime('%H:%M:%S')}–{b['last_seen'].strftime('%H:%M:%S')})"
        )

    if ip in _saf_by_ip:
        s = _saf_by_ip[ip]
        parts.append(
            f"success_after_failures: {s['prior_failures']} failures then login "
            f"as '{s['username']}' at {s['login_time'].strftime('%H:%M:%S')}"
        )

    if ip in _enum_by_ip:
        e = _enum_by_ip[ip]
        parts.append(
            f"many_usernames: {e['unique_usernames']} distinct usernames probed "
            f"({', '.join(e['usernames'])})"
        )

    return " | ".join(parts)


def _build_alert(ip):
    """Return a complete alert dict for one flagged IP."""
    rep = reputation.get(ip, {})  # empty dict if IP not in reputation feed

    # Reputation values — 'no data' when the IP is absent from the feed
    abuse_confidence = rep.get("abuse_confidence", "no data")
    reports          = rep.get("reports",          "no data")

    # Which rules fired
    rules = []
    if ip in _burst_by_ip:  rules.append("rule_burst")
    if ip in _saf_by_ip:    rules.append("rule_success_after_failures")
    if ip in _enum_by_ip:   rules.append("rule_many_usernames")

    return {
        "ip":               ip,
        "rules_triggered":  ", ".join(rules),
        "evidence":         _build_evidence(ip),
        "abuse_confidence": abuse_confidence,
        "reports":          reports,
    }


# Build one alert per flagged IP, sorted alphabetically
alerts = [_build_alert(ip) for ip in sorted(all_flagged_ips)]

print(f"Alerts built for {len(alerts)} IP(s)")

Alerts built for 3 IP(s)


### 3.3 — Severity scoring and verdict

Each alert is scored using the point table from the section header. The resulting severity label and verdict (including recommended action and MITRE ATT&CK technique) are appended to each alert dict.

In [50]:
def _compute_severity(alert):
    """
    Award points based on which rules fired and the abuse_confidence score.
    Return (points, label) where label is 'High', 'Medium', or 'Low'.
    """
    points = 0
    rules = alert["rules_triggered"]

    if "rule_burst"                   in rules: points += 2
    if "rule_success_after_failures"  in rules: points += 4
    if "rule_many_usernames"          in rules: points += 2

    # Reputation bonus — only when we have a numeric value
    ac = alert["abuse_confidence"]
    if isinstance(ac, (int, float)):
        if ac >= 75:          points += 2
        elif 30 <= ac < 75:   points += 1

    if points >= 6:   label = "High"
    elif points >= 3: label = "Medium"
    else:             label = "Low"

    return points, label


def _compute_verdict(alert):
    """
    Derive a recommended action and MITRE ATT&CK technique(s) from the
    alert's rule set and severity.

    MITRE mapping:
      T1110 Brute Force           — always present when any failure-based rule fires
      T1078 Valid Accounts        — added when a login succeeded (SAF rule)
    """
    rules = alert["rules_triggered"]
    has_saf   = "rule_success_after_failures" in rules
    has_burst = "rule_burst"                  in rules
    has_enum  = "rule_many_usernames"         in rules

    # MITRE techniques
    mitre = ["T1110 Brute Force"]
    if has_saf:
        mitre.append("T1078 Valid Accounts")

    # Recommended action
    if has_saf:
        action = (
            "SUCCESS AFTER FAILURES — possible credential compromise. "
            "Block IP immediately, revoke session, escalate to L2 for forensic review."
        )
    elif has_burst:
        action = (
            "BURST WITHOUT SUCCESS — brute-force attempt, no confirmed compromise. "
            "Block IP, increase fail2ban sensitivity, continue monitoring."
        )
    elif has_enum:
        action = (
            "USERNAME ENUMERATION — credential-stuffing/scan pattern. "
            "Monitor closely; consider geo-block or rate-limit on this IP."
        )
    else:
        action = "Review manually."

    return action, ", ".join(mitre)


# Enrich every alert with severity and verdict in-place
for alert in alerts:
    points, label        = _compute_severity(alert)
    action, mitre        = _compute_verdict(alert)
    alert["severity_pts"] = points
    alert["severity"]     = label
    alert["mitre"]        = mitre
    alert["verdict"]      = action

print("Severity and verdict assigned to all alerts.")

Severity and verdict assigned to all alerts.


### 3.4 — Alert table

Display the final enriched alert DataFrame, sorted by descending severity (High → Medium → Low).

In [51]:
# Build the display DataFrame
alert_df = pd.DataFrame(alerts, columns=[
    "ip", "rules_triggered", "evidence",
    "abuse_confidence", "reports",
    "severity_pts", "severity", "mitre", "verdict",
])

# Sort High → Medium → Low for readability
_severity_order = {"High": 0, "Medium": 1, "Low": 2}
alert_df["_sort"] = alert_df["severity"].map(_severity_order)
alert_df = alert_df.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

# Print a concise console view
print("=" * 70)
print(f"{'ALERT TABLE':^70}")
print("=" * 70)
for _, row in alert_df.iterrows():
    print(f"  IP          : {row['ip']}")
    print(f"  Severity    : {row['severity']} ({row['severity_pts']} pts)")
    print(f"  Rules       : {row['rules_triggered']}")
    print(f"  Evidence    : {row['evidence']}")
    print(f"  Reputation  : abuse_confidence={row['abuse_confidence']}  reports={row['reports']}")
    print(f"  MITRE       : {row['mitre']}")
    print(f"  Verdict     : {row['verdict']}")
    print("-" * 70)

# Also show as a DataFrame for notebook convenience
alert_df

                             ALERT TABLE                              
  IP          : 198.51.100.77
  Severity    : High (8 pts)
  Rules       : rule_burst, rule_success_after_failures
  Evidence    : burst: 14 failures in 5-min window (14:30:00–14:31:49) | success_after_failures: 14 failures then login as 'root' at 14:32:20
  Reputation  : abuse_confidence=78  reports=140
  MITRE       : T1110 Brute Force, T1078 Valid Accounts
  Verdict     : SUCCESS AFTER FAILURES — possible credential compromise. Block IP immediately, revoke session, escalate to L2 for forensic review.
----------------------------------------------------------------------
  IP          : 192.0.2.10
  Severity    : Medium (3 pts)
  Rules       : rule_many_usernames
  Evidence    : many_usernames: 9 distinct usernames probed (ftpuser, git, guest, oracle, pi, postgres, test, ubuntu, user)
  Reputation  : abuse_confidence=35  reports=22
  MITRE       : T1110 Brute Force
  Verdict     : USERNAME ENUMERATION — credential

,ip,rules_triggered,evidence,abuse_confidence,reports,severity_pts,severity,mitre,verdict
0,198.51.100.77,"rule_burst, rule_success_after_failures",burst: 14 failures in 5-min window (14:30:00–1...,78,140,8,High,"T1110 Brute Force, T1078 Valid Accounts",SUCCESS AFTER FAILURES — possible credential c...
1,192.0.2.10,rule_many_usernames,many_usernames: 9 distinct usernames probed (f...,35,22,3,Medium,T1110 Brute Force,USERNAME ENUMERATION — credential-stuffing/sca...
2,203.0.113.45,rule_burst,burst: 40 failures in 5-min window (02:14:00–0...,92,310,4,Medium,T1110 Brute Force,"BURST WITHOUT SUCCESS — brute-force attempt, n..."


### 3.5 — IPs with failed logins but no rule triggered

Some IPs produced at least one `failed_password` or `invalid_user` event yet never crossed any rule threshold. These are worth noting explicitly so an analyst knows they were reviewed and consciously dismissed.

In [52]:
# Collect all IPs that had at least one failure-type event
_failure_event_types = {"failed_password", "invalid_user"}
_ips_with_failures = set(
    df.loc[df["event_type"].isin(_failure_event_types), "source_ip"].unique()
)

# Remove the ones already captured by a rule
no_rule_ips = _ips_with_failures - all_flagged_ips

print(f"IPs with failures but no rule triggered: {sorted(no_rule_ips)}")
print()

# Show failure counts per non-flagged IP for context
_no_rule_rows = df[
    df["event_type"].isin(_failure_event_types)
    & df["source_ip"].isin(no_rule_ips)
]
for ip, grp in _no_rule_rows.groupby("source_ip"):
    users = sorted(grp["username"].unique().tolist())
    print(f"  {ip:<18}  failures={len(grp)}  usernames={users}")

IPs with failures but no rule triggered: ['10.0.0.15', '10.0.0.22', '10.0.0.31', '10.0.0.5']

  10.0.0.15           failures=3  usernames=['priya']
  10.0.0.22           failures=1  usernames=['rahul']
  10.0.0.31           failures=1  usernames=['meera']
  10.0.0.5            failures=1  usernames=['dev_admin']


> **Note — why these IPs have no alert:**
>
> **`10.0.0.15`** — 3 failed attempts for user `priya` between 09:41:05 and 09:41:29 (~24 seconds), followed by a successful login at 09:42:10.  
> The burst threshold requires ≥ 10 failures in 5 minutes (only 3 seen), the SAF rule requires ≥ 5 prior failures before a success (only 3 seen), and it probed only 1 username so the enumeration rule cannot fire.  
> The pattern — 3 quick misses then success for the *same username* from an **internal RFC-1918 address** — is consistent with a **mistyped password** (e.g., wrong capitalisation on first keystroke).  
> No alert is warranted; the session is legitimate. The IP should remain on a watch list for today but requires no escalation.

---
## Section 4 — Incident Report

This section writes two output files:

| File | Format | Content |
|---|---|---|
| `output/incident_report.md` | Markdown | Human-readable report with alert details, verdict, and notes |
| `output/incident_report.csv` | CSV | Machine-readable flat table of all alert fields |

The `output/` directory is created automatically if it does not already exist.  
Both files are overwritten on each run.

### 4.1 — Write `output/incident_report.md`

In [53]:
import os
from datetime import timezone

# Create the output directory if it does not exist
os.makedirs("output", exist_ok=True)

MD_PATH  = "output/incident_report.md"
CSV_PATH = "output/incident_report.csv"

# ── Build the Markdown string ────────────────────────────────────────────────
_md_lines = [
    "# Incident Report — Auth Log Triage",
    "",
    f"**Generated:** 2026-10-18  ",
    f"**Log file:** `data/sample_auth.log`  ",
    f"**Reputation feed:** `data/mock_reputation.json`  ",
    "",
    "---",
    "",
    "## Alerts",
    "",
]

for _, row in alert_df.iterrows():
    _md_lines += [
        f"### {row['ip']} — {row['severity']} ({row['severity_pts']} pts)",
        "",
        f"| Field | Value |",
        f"|---|---|",
        f"| **Rules triggered** | {row['rules_triggered']} |",
        f"| **Evidence** | {row['evidence']} |",
        f"| **abuse_confidence** | {row['abuse_confidence']} |",
        f"| **reports** | {row['reports']} |",
        f"| **MITRE ATT&CK** | {row['mitre']} |",
        f"| **Verdict** | {row['verdict']} |",
        "",
    ]

_md_lines += [
    "---",
    "",
    "## IPs with failures — no rule triggered",
    "",
]

for ip in sorted(no_rule_ips):
    _grp = _no_rule_rows[_no_rule_rows["source_ip"] == ip]
    _users = sorted(_grp["username"].unique().tolist())
    _count = len(_grp)
    _md_lines.append(
        f"- **{ip}** — {_count} failure(s), username(s): {', '.join(_users)}. "
        "Pattern consistent with mistyped password (internal address, same username, "
        "< 5 failures, followed by successful login). No escalation required."
    )

_md_lines += ["", "---", "", "*End of report.*", ""]

# Write the Markdown file
_md_text = "\n".join(_md_lines)
with open(MD_PATH, "w", encoding="utf-8") as _fh:
    _fh.write(_md_text)

print(f"Markdown report written → {MD_PATH}  ({len(_md_lines)} lines)")

Markdown report written → output/incident_report.md  (56 lines)


### 4.2 — Write `output/incident_report.csv`

In [54]:
# Write the CSV — all columns, no index, UTF-8 with BOM for Excel compatibility
alert_df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

print(f"CSV report written → {CSV_PATH}  ({len(alert_df)} data rows + header)")

CSV report written → output/incident_report.csv  (3 data rows + header)


### 4.3 — Print the full report to stdout

In [55]:
# Print the Markdown report to stdout for inline notebook review
print(_md_text)

# Incident Report — Auth Log Triage

**Generated:** 2026-10-18  
**Log file:** `data/sample_auth.log`  
**Reputation feed:** `data/mock_reputation.json`  

---

## Alerts

### 198.51.100.77 — High (8 pts)

| Field | Value |
|---|---|
| **Rules triggered** | rule_burst, rule_success_after_failures |
| **Evidence** | burst: 14 failures in 5-min window (14:30:00–14:31:49) | success_after_failures: 14 failures then login as 'root' at 14:32:20 |
| **abuse_confidence** | 78 |
| **reports** | 140 |
| **MITRE ATT&CK** | T1110 Brute Force, T1078 Valid Accounts |
| **Verdict** | SUCCESS AFTER FAILURES — possible credential compromise. Block IP immediately, revoke session, escalate to L2 for forensic review. |

### 192.0.2.10 — Medium (3 pts)

| Field | Value |
|---|---|
| **Rules triggered** | rule_many_usernames |
| **Evidence** | many_usernames: 9 distinct usernames probed (ftpuser, git, guest, oracle, pi, postgres, test, ubuntu, user) |
| **abuse_confidence** | 35 |
| **reports** | 22 |
| **M

### 4.4 — Self-check (assertions)

Validate that the alert table produced the expected severity labels for the three flagged IPs, and confirm that `10.0.0.15` (the mistyped-password case) produced no alert.

In [56]:
# Build a quick ip→severity look-up from the alert DataFrame
_sev_map = dict(zip(alert_df["ip"], alert_df["severity"]))

# ── Assertions ───────────────────────────────────────────────────────────────

# 198.51.100.77: burst + SAF + high abuse_confidence → 2+4+2 = 8 pts → High
assert _sev_map.get("198.51.100.77") == "High", (
    f"Expected High for 198.51.100.77, got {_sev_map.get('198.51.100.77')}"
)

# 203.0.113.45: burst only + high abuse_confidence → 2+2 = 4 pts → Medium
assert _sev_map.get("203.0.113.45") == "Medium", (
    f"Expected Medium for 203.0.113.45, got {_sev_map.get('203.0.113.45')}"
)

# 192.0.2.10: many_usernames + mid abuse_confidence → 2+1 = 3 pts → Medium
assert _sev_map.get("192.0.2.10") == "Medium", (
    f"Expected Medium for 192.0.2.10, got {_sev_map.get('192.0.2.10')}"
)

# 10.0.0.15: only 3 failures (below every threshold) — must NOT appear in alerts
assert "10.0.0.15" not in _sev_map, (
    "10.0.0.15 should not have an alert (mistyped-password pattern, below all thresholds)"
)

print("All self-check assertions passed.")

All self-check assertions passed.
